# Дискретно-событийная SIR модель

Данный скрипт реализует стохастическую дискретно-событийную модель
распространения инфекции SIR (Susceptible-Infected-Recovered).
Модель использует ConcurrentSim для управления событиями и агентов
с индивидуальным поведением.

## Подключение модулей

In [ ]:
using DrWatson
@quickactivate "project"
include(srcdir("sir_model.jl"))
using Random, StatsPlots, BenchmarkTools

## Параметры модели

- `tmax` - максимальное время симуляции
- `u0` - начальные численности [S, I, R]
- `p` - параметры модели [β, c, γ]
  - β - вероятность заражения при контакте
  - c - частота контактов
  - γ - интенсивность выздоровления

In [ ]:
tmax = 40.0
u0 = [990, 10, 0]  # S, I, R
p = [0.05, 10.0, 0.25]  # β, c, γ

Фиксируем seed для воспроизводимости результатов

In [ ]:
Random.seed!(1234)

Выводим базовое репродуктивное число

In [ ]:
println("R₀ = $(round(p[1]*p[2]/p[3], digits=2))")

## Запуск модели

Создаём экземпляр модели, активируем процессы и запускаем симуляцию.

In [ ]:
des_model = MakeSIRModel(u0, p)
activate(des_model)
sir_run(des_model, tmax)
data_des = out(des_model)

println("Симуляция завершена. Собрано событий: $(length(data_des.t))")

## Визуализация результатов

Строим график динамики эпидемии:
- S(t) - восприимчивые (синий)
- I(t) - инфицированные (красный)
- R(t) - переболевшие (зелёный)

In [ ]:
@df data_des plot(
    :t,
    [:S :I :R],
    labels = ["S" "I" "R"],
    xlab = "Время",
    ylab = "Численность",
    title = "Дискретно-событийная SIR модель",
)

savefig(plotsdir("sir_des.png"))
display(p)
println("График сохранён в plots/sir_des.png")

## Финальная статистика

Выводим итоговые значения численностей и параметры эпидемии.

In [ ]:
println("\n=== Финальная статистика ===")
println("Время симуляции: $(data_des.t[end])")
println("Восприимчивые (S): $(data_des.S[end])")
println("Инфицированные (I): $(data_des.I[end])")
println("Переболевшие (R): $(data_des.R[end])")

Пик эпидемии

In [ ]:
max_I = maximum(data_des.I)
peak_time = data_des.t[argmax(data_des.I)]
println("\nПик эпидемии:")
println("  Максимальное число инфицированных: $max_I")
println("  Время достижения пика: t = $(round(peak_time, digits=2))")

## Интерпретация результатов

На графике видно:
1. Сначала инфицированные растут за счёт заражения восприимчивых
2. Достигается пик эпидемии
3. Инфицированные снижаются, переболевшие растут
4. В отличие от детерминированной модели, наблюдаются стохастические флуктуации